# 🧠 Tactile SDF Reconstruction — Google Colab

This notebook trains a **PointNet + SIREN** model using sparse tactile contacts.

**Fully self-contained pipeline:**
1. Downloads grasp dataset (contacts) from Hugging Face
2. Downloads precomputed SDF cache from HF *(or recomputes from Objaverse if cache is missing)*
3. Trains the model on GPU
4. Saves results to Google Drive

In [ ]:
# @title 🛠️ Step 1: Setup & Dependencies
!pip install -q trimesh rtree scikit-image scikit-learn tqdm huggingface_hub objaverse
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ None — go to Runtime > Change runtime type > T4 GPU!'}")

In [ ]:
# @title 📂 Step 2: Clone Code & Download Grasp Dataset
import os
from huggingface_hub import snapshot_download

ROOT = "/content/tactile_project"
os.makedirs(ROOT, exist_ok=True)

# Clone tactile-sdf code
if not os.path.exists(f'{ROOT}/code'):
    !git clone https://github.com/635jack/tactile-sdf.git {ROOT}/code
else:
    !cd {ROOT}/code && git pull
print("✅ Code ready")

# Download grasp dataset + SDF cache from HF
HF_REPO = "jack635/grasp-dataset-curated"
DATASET_DIR = f"{ROOT}/grasp-dataset"
if not os.path.exists(f'{DATASET_DIR}/dataset_index.json'):
    print(f"📥 Downloading dataset from {HF_REPO}...")
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=DATASET_DIR)
else:
    print("✅ Dataset already downloaded")

SDF_CACHE_DIR = f"{DATASET_DIR}/sdf_cache"
n_sdf = len(os.listdir(SDF_CACHE_DIR)) if os.path.exists(SDF_CACHE_DIR) else 0
print(f"📦 SDF cache: {n_sdf} objects found")

In [ ]:
# @title 📐 Step 3: SDF Preprocessing (auto-skipped if cache exists on HF)
# If the HF SDF cache has all objects, this cell does nothing.
# If some are missing (e.g. new objects added), it recomputes them from Objaverse.
import json, shutil, objaverse

MESH_DIR = f"{ROOT}/data/objaverse"
os.makedirs(MESH_DIR, exist_ok=True)

# Check which objects are missing from the SDF cache
with open(f'{DATASET_DIR}/dataset_index.json') as f:
    index = json.load(f)
all_objects = [obj['mesh'] for obj in index['objects']]
cached = set(f.replace('.npz','') for f in os.listdir(SDF_CACHE_DIR)) if os.path.exists(SDF_CACHE_DIR) else set()
missing = [o for o in all_objects if o not in cached]

if not missing:
    print(f"✅ All {len(all_objects)} objects already in SDF cache — skipping preprocessing!")
else:
    print(f"⚙️  {len(missing)} objects missing from SDF cache — downloading from Objaverse and computing...")

    # Download only the missing GLBs from Objaverse
    with open(f'{DATASET_DIR}/uid_mapping.json') as f:
        uid_mapping = json.load(f)
    missing_uids = {uid_mapping[o]: o for o in missing if o in uid_mapping}

    objects = objaverse.load_objects(uids=list(missing_uids.keys()))
    for uid, mesh_name in missing_uids.items():
        if uid in objects:
            shutil.copy(objects[uid], f"{MESH_DIR}/{mesh_name}.glb")

    # Run preprocessing only on missing objects
    !python {ROOT}/code/preprocess_sdf.py \
        --glb_dir {MESH_DIR} \
        --dataset_dir {DATASET_DIR} \
        --sdf_cache_dir {SDF_CACHE_DIR} \
        --n_points 50000
    print("✅ Preprocessing done")

In [ ]:
# @title 🚀 Step 4: Train!
import sys
!{sys.executable} {ROOT}/code/train.py \
    --epochs 300 \
    --batch_size 16 \
    --dataset_dir {DATASET_DIR} \
    --sdf_cache_dir {SDF_CACHE_DIR} \
    --output_dir {ROOT}/runs \
    --eval_every 10 \
    --save_every 50

In [ ]:
# @title 📊 Step 5: Visualize Results
import matplotlib.pyplot as plt, matplotlib.image as mpimg, glob

runs = sorted(glob.glob(f'{ROOT}/runs/*'))
if runs:
    latest_run = runs[-1]
    print(f"Latest run: {latest_run}")
    img_path = f"{latest_run}/training_curves.png"
    if os.path.exists(img_path):
        plt.figure(figsize=(15, 10))
        plt.imshow(mpimg.imread(img_path))
        plt.axis('off')
        plt.show()
else:
    print("No training runs found.")

In [ ]:
# @title 💾 Step 6 (Optional): Save results to Google Drive
from google.colab import drive
drive.mount('/content/drive')

runs = sorted(glob.glob(f'{ROOT}/runs/*'))
if runs:
    import shutil
    dest = '/content/drive/MyDrive/tactile_sdf_results'
    os.makedirs(dest, exist_ok=True)
    latest_run = runs[-1]
    shutil.copytree(latest_run, f"{dest}/{os.path.basename(latest_run)}", dirs_exist_ok=True)
    print(f"✅ Saved to {dest}")